In [2]:
import pandas as pd

# 1. Leer el CSV
df = pd.read_csv(
    "../data/viajeros_pernoctaciones_2074_11_05.csv",
    sep=";"
)

# 2. Convertir la columna Total a número
df["Total_num"] = (
    df["Total"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df["Total_num"] = pd.to_numeric(df["Total_num"], errors="coerce")

# 3. Quedarse solo con comunidades autónomas
df_ccaa = df[
    df["Comunidades y Ciudades Autónomas"].notna()
    & df["Provincias"].isna()
].copy()

# 4. Crear columna limpia de residencia
df_ccaa["Residencia"] = df_ccaa["Residencia: Nivel 2"].fillna("Total")

# 5. Pivotar viajeros y pernoctaciones
df_pivot = df_ccaa.pivot_table(
    index=[
        "Comunidades y Ciudades Autónomas",
        "Periodo",
        "Residencia"
    ],
    columns="Viajeros y pernoctaciones",
    values="Total_num",
    aggfunc="sum"
).reset_index()

# 6. Limpiar nombre del eje de columnas
df_pivot.columns.name = None

# 7. Ver resultado
df_pivot.head()

,Comunidades y Ciudades Autónomas,Periodo,Residencia,Pernoctaciones,Viajero
0,01 Andalucía,1999M01,Residentes en España,709064.0,300094.0
1,01 Andalucía,1999M01,Residentes en el Extranjero,802728.0,183948.0
2,01 Andalucía,1999M01,Total,1511792.0,484042.0
3,01 Andalucía,1999M02,Residentes en España,886785.0,388725.0
4,01 Andalucía,1999M02,Residentes en el Extranjero,942113.0,232301.0


In [8]:
# Quedarse solo con datos entre 2022 y 2026
df_pivot = df_pivot[
    df_pivot["Periodo"].str[:4].astype(int).between(2022, 2026)
]

In [9]:
df_pivot

,Comunidades y Ciudades Autónomas,Periodo,Residencia,Pernoctaciones,Viajero
16524,"17 Rioja, La",2022M01,Residentes en España,26860.0,16581.0
15544,16 País Vasco,2022M01,Residentes en el Extranjero,71468.0,39914.0
6715,07 Castilla y León,2022M01,Residentes en el Extranjero,53007.0,29288.0
16525,"17 Rioja, La",2022M01,Residentes en el Extranjero,4205.0,2273.0
11620,12 Galicia,2022M01,Residentes en el Extranjero,41212.0,17896.0
...,...,...,...,...,...
6866,07 Castilla y León,2026M03,Total,642005.0,371333.0
11771,12 Galicia,2026M03,Total,505193.0,276405.0
11770,12 Galicia,2026M03,Residentes en el Extranjero,118429.0,71318.0
2942,"03 Asturias, Principado de",2026M03,Total,207802.0,113361.0


In [11]:
df_pivot["Periodo_fecha"] = pd.to_datetime(
    df_pivot["Periodo"].str.replace("M", "-"),
    format="%Y-%m"
)

In [12]:
df_pivot

,Comunidades y Ciudades Autónomas,Periodo,Residencia,Pernoctaciones,Viajero,Periodo_fecha
16524,"17 Rioja, La",2022M01,Residentes en España,26860.0,16581.0,2022-01-01
15544,16 País Vasco,2022M01,Residentes en el Extranjero,71468.0,39914.0,2022-01-01
6715,07 Castilla y León,2022M01,Residentes en el Extranjero,53007.0,29288.0,2022-01-01
16525,"17 Rioja, La",2022M01,Residentes en el Extranjero,4205.0,2273.0,2022-01-01
11620,12 Galicia,2022M01,Residentes en el Extranjero,41212.0,17896.0,2022-01-01
...,...,...,...,...,...,...
6866,07 Castilla y León,2026M03,Total,642005.0,371333.0,2026-03-01
11771,12 Galicia,2026M03,Total,505193.0,276405.0,2026-03-01
11770,12 Galicia,2026M03,Residentes en el Extranjero,118429.0,71318.0,2026-03-01
2942,"03 Asturias, Principado de",2026M03,Total,207802.0,113361.0,2026-03-01


In [16]:
# Valores únicos de Comunidades y Ciudades Autónomas
df_pivot["Comunidades y Ciudades Autónomas"].dropna().unique()

array(['17 Rioja, La', '16 País Vasco', '07 Castilla y León',
       '12 Galicia', '10 Comunitat Valenciana', '13 Madrid, Comunidad de',
       '19 Melilla', '08 Castilla - La Mancha', '06 Cantabria',
       '09 Cataluña', '04 Balears, Illes', '14 Murcia, Región de',
       '01 Andalucía', '18 Ceuta', '03 Asturias, Principado de',
       '02 Aragón', '15 Navarra, Comunidad Foral de', '11 Extremadura',
       '05 Canarias'], dtype=object)

In [22]:
# Eliminar códigos numéricos al inicio
df_pivot.loc[:, "Comunidades y Ciudades Autónomas"] = (
    df_pivot["Comunidades y Ciudades Autónomas"]
    .str.replace(r"^\d+\s+", "", regex=True)
)

#compruebo:
# Valores únicos de Comunidades y Ciudades Autónomas
df_pivot["Comunidades y Ciudades Autónomas"].dropna().unique()

array(['Rioja, La', 'País Vasco', 'Castilla y León', 'Galicia',
       'Castilla - La Mancha', 'Cantabria', 'Balears, Illes', 'Cataluña',
       'Murcia, Región de', 'Comunitat Valenciana', 'Andalucía', 'Ceuta',
       'Asturias, Principado de', 'Aragón', 'Extremadura', 'Melilla',
       'Navarra, Comunidad Foral de', 'Madrid, Comunidad de', 'Canarias'],
      dtype=object)

In [17]:
# Valores únicos de Periodo
df_pivot["Periodo"].dropna().unique()

array(['2022M01', '2022M02', '2022M03', '2022M04', '2022M05', '2022M06',
       '2022M07', '2022M08', '2022M09', '2022M10', '2022M11', '2022M12',
       '2023M01', '2023M02', '2023M03', '2023M04', '2023M05', '2023M06',
       '2023M07', '2023M08', '2023M09', '2023M10', '2023M11', '2023M12',
       '2024M01', '2024M02', '2024M03', '2024M04', '2024M05', '2024M06',
       '2024M07', '2024M08', '2024M09', '2024M10', '2024M11', '2024M12',
       '2025M01', '2025M02', '2025M03', '2025M04', '2025M05', '2025M06',
       '2025M07', '2025M08', '2025M09', '2025M10', '2025M11', '2025M12',
       '2026M01', '2026M02', '2026M03'], dtype=object)

In [18]:
# Valores únicos de Residencia
df_pivot["Residencia"].dropna().unique()

array(['Residentes en España', 'Residentes en el Extranjero', 'Total'],
      dtype=object)

In [19]:
# Eliminar filas donde Residencia = Total
df_pivot = df_pivot[
    df_pivot["Residencia"] != "Total"
]

In [20]:
# compruebo
# Valores únicos de Residencia
df_pivot["Residencia"].dropna().unique()

array(['Residentes en España', 'Residentes en el Extranjero'],
      dtype=object)